In [11]:
import torch                  #PyTorch 深度学习核心库，负责张量运算、模型运行
import torch.nn as nn         #PyTorch 神经网络模块，包含卷积、全连接、激活函数等网络层
from torch.utils.data import DataLoader  #数据加载器：批量读取数据、分批送入模型
from torchvision import datasets, transforms  #视觉数据集 + 图像预处理工具
import numpy as np            #数值计算库，用于统计每一类样本数量、正确预测数量

In [12]:
#重新定义卷积神经网络模型(和训练代码一模一样）
class ConeModel(nn.Module):
    #网络初始化函数
    def __init__(self):
        #继承父类 nn.Module 的所有属性和方法
        super(ConeModel, self).__init__()

        #卷积层 + 池化层
        #卷积层：输入3通道，输出16通道特征图，卷积核3×3，边缘补1个像素
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        #最大池化层：池化窗口2×2，步长2，缩小图像尺寸、减少计算量、提取关键特征
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)   #第二层卷积
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)   #第三层卷积
        self.pool3 = nn.MaxPool2d(2, 2)

        self.conv4 = nn.Conv2d(64, 128, 3, padding=1)  #第四层卷积
        self.pool4 = nn.MaxPool2d(2, 2)

        #全连接层，输入维度：128 * 8 * 8
        #推导：原图尺寸 128×128，经过4次池化(每次缩小1/2) → 128/(2^4) = 8
        #最终特征图：128通道 × 8高 × 8宽
        self.fc1 = nn.Linear(128 * 8 * 8, 256)  #第一层全连接，特征降维
        self.fc2 = nn.Linear(256, 3)            #最后一层全连接：输出3个类别（对应3类锥桶）

        #激活函数 ReLU：给网络引入非线性，让模型能学习复杂图像特征
        self.relu = nn.ReLU()

    #前向传播函数：定义数据在网络中的流动顺序
    def forward(self, x):
        #流程：卷积 → 激活函数 → 池化，重复4组卷积块
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = self.pool3(self.relu(self.conv3(x)))
        x = self.pool4(self.relu(self.conv4(x)))

        # x.view：将多维特征图展平为一维向量，因为全连接层只接受一维数据
        # x.size(0) 代表批次大小，-1 自动计算剩余维度
        x = x.view(x.size(0), -1)

        #全连接层 + 激活函数，最后输出预测结果
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [13]:
#定义图像预处理规则（和训练集预处理一致）
test_transform = transforms.Compose([
    transforms.Resize((128, 128)),    #统一把所有图片缩放到 128×128
    transforms.ToTensor(),            #将PIL图片 → PyTorch张量，像素值从 [0,255] 归一化到 [0,1]
    #标准化：均值0.5、方差0.5，把像素值从 [0,1] 映射到 [-1, 1]，提升模型稳定性
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

In [14]:
#加载两个测试数据集 test1、test2
#自动读取文件夹分类（文件夹名 = 类别名），分类任务专用
#加载第一个测试集
test1_data = datasets.ImageFolder("./dataset/test1", transform=test_transform)
#批量加载数据，批次大小16，测试集不需要打乱顺序
test1_loader = DataLoader(test1_data, batch_size=16, shuffle=False)

#加载第二个独立测试集 test2
test2_data = datasets.ImageFolder("./dataset/test2", transform=test_transform)
test2_loader = DataLoader(test2_data, batch_size=16, shuffle=False)

#获取类别名称列表：按文件夹顺序得到3个锥桶的类别名（用于打印结果）
class_name = list(test1_data.class_to_idx.keys())

In [15]:
#配置运行设备 + 加载训练好的模型权重
#自动判断设备：有NVIDIA显卡(CUDA)就用GPU加速，没有就用CPU运行
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#创建网络模型实例
model = ConeModel().to(device)

#加载训练保存的模型权重文件 cone_model.pth（网络结构须和训练时一致）
model.load_state_dict(torch.load("cone_model.pth"))

#切换为评估模式，关闭 Dropout、BatchNorm 等训练专用层，保证测试结果稳定
model.eval()

ConeModel(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=8192, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=3, bias=True)
  (relu): ReLU()
)

In [16]:
#自定义测试函数
#参数说明：
#   loader：数据集加载器（test1_loader / test2_loader）
#   name：数据集名称（用于打印区分）
#功能：统计总样本、正确样本、每一类样本的准确率
def test_all(loader, name):
    total_num = 0                #统计：当前数据集 总样本数量
    right_num = 0                #统计：模型预测正确的样本数量
    class_total = np.zeros(3)    #数组：记录3个类别各自的总样本数，初始化为0
    class_right = np.zeros(3)    #数组：记录3个类别各自预测正确的样本数，初始化为0

    #测试阶段不需要反向传播、不需要计算梯度，节省显存、加快运行速度
    with torch.no_grad():
        #循环遍历数据加载器中的每一个批次（一批16张图）
        for images, labels in loader:
            #把图像、标签数据迁移到对应设备(GPU/CPU)，和模型设备保持一致
            images = images.to(device)
            labels = labels.to(device)

            #模型前向推理：输入图片，输出预测分值
            outputs = model(images)
            #torch.max：取出每行最大值（预测分值）和对应索引（预测类别）
            #_是占位符，丢弃分值；pred 保存预测的类别编号
            _, pred = torch.max(outputs, 1)

            #累加当前批次的样本总数
            total_num += labels.size(0)
            #(pred == labels) 得到布尔数组，sum() 统计预测正确的数量
            right_num += (pred == labels).sum().item()

            #逐个遍历当前批次的每一张图片，统计每个类别的正误
            for i in range(len(labels)):
                lab = labels[i].item()   #取出真实标签（真实类别）
                pre = pred[i].item()     #取出模型预测标签（预测类别）
                class_total[lab] += 1    #对应类别总数量 +1
                if lab == pre:           #如果预测和真实标签一致
                    class_right[lab] += 1 #对应类别正确数量 +1

    #打印测试结果
    print(f"===== {name} 测试结果 =====")
    #计算整体准确率：正确数 / 总数 * 100，保留2位小数
    print(f"整体准确率：{100 * right_num / total_num:.2f} %")

    #循环打印每一个类别的单独准确率
    for i in range(3):
        acc = 100 * class_right[i] / class_total[i]
        print(f"{class_name[i]} 准确率：{acc:.2f} %")

    #区分不同数据集结果
    print("----------------------------------------")

In [17]:
#调用测试函数，分别对两个测试集执行测试
test_all(test1_loader, "测试集 test1")
test_all(test2_loader, "测试集 test2")

===== 测试集 test1 测试结果 =====
整体准确率：98.16 %
blue 准确率：97.59 %
red 准确率：98.77 %
yellow 准确率：98.11 %
----------------------------------------
===== 测试集 test2 测试结果 =====
整体准确率：93.57 %
blue 准确率：98.00 %
red 准确率：87.76 %
yellow 准确率：95.12 %
----------------------------------------
